# BlueShift: NASA Environmental + PSA Fisheries Data Integration

Integrate quarterly provincial NASA chlorophyll-a/SST measurements with PSA fisheries production records for Mindanao and export one clean CSV for EDA.

## Final output columns

1. `Year`
2. `Quarter`
3. `Province`
4. `Species`
5. `Chlorophyll-A Concentration (mg/m³)`
6. `Sea Surface Temperature (°C)`
7. `Yield (Metric Tons)`

**Grain:** one row per Year × Quarter × Province × Species.

### Important decisions
- Only quarterly records are retained. Annual PSA records and monthly NASA records are not compatible with the quarterly integration.
- NASA measurements are joined using `Year`, `Quarter`, and `Province`.
- `ALL SPECIES` is excluded because it is an aggregate total. Keeping it alongside individual species would mix totals and species-level observations and could lead to double counting.
- Parenthetical local-language names are removed from PSA species labels. `P. Vannamei` is standardized to `Pacific White Shrimp`.
- For duplicate PSA keys, a `reported` record takes priority over an `unavailable` record. `below_reporting_precision` remains a numeric zero because PSA explicitly reports it as `0.00`.
- `unavailable` production remains missing (`NaN`); it is **not** converted to zero or imputed.
- The join uses the overlapping quarters available in both sources. Quarters without NASA observations cannot be included in this integrated dataset.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Configure file paths

In [ ]:
# Update these paths if your CSV files are stored elsewhere.
NASA_FILE = Path("../data/processed/nasa/nasa_provincial_quarterly.csv")
PSA_FILE = Path("../data/processed/psa/psa_mindanao_production.csv")

OUTPUT_FILE = Path("../data/processed/blueshift_dataset.csv")

# Check that both input files exist before proceeding.
assert NASA_FILE.exists(), f"NASA file not found: {NASA_FILE.resolve()}"
assert PSA_FILE.exists(), f"PSA file not found: {PSA_FILE.resolve()}"

## 3. Load the source datasets

In [ ]:
nasa = pd.read_csv(NASA_FILE)
psa = pd.read_csv(PSA_FILE)

print("NASA dataset:", nasa.shape)
print("PSA dataset:", psa.shape)

display(nasa.head())
display(psa.head())

NASA dataset: (9192, 15)
PSA dataset: (209375, 13)


,period_type,period,year,quarter,month,province_id,province,chlor_a_mean,chlor_a_valid_pixels,chlor_a_mask_pixels,sst_mean,sst_valid_pixels,sst_mask_pixels,chlor_a_months_observed,sst_months_observed
0,monthly,2002-07,2002,Q3,7.0000,1,Zamboanga del Norte,0.2152,271,274,29.5965,274,274,NaN,NaN
1,monthly,2002-08,2002,Q3,8.0000,1,Zamboanga del Norte,0.2919,262,274,29.2575,274,274,NaN,NaN
2,monthly,2002-09,2002,Q3,9.0000,1,Zamboanga del Norte,0.2317,210,274,29.3639,273,274,NaN,NaN
3,quarterly,2002-Q3,2002,Q3,NaN,1,Zamboanga del Norte,0.2463,743,822,29.4060,821,822,3.0000,3.0000
4,monthly,2002-10,2002,Q4,10.0000,1,Zamboanga del Norte,0.3047,273,274,30.1601,274,274,NaN,NaN


,Species,species,Geolocation,geolocation_label,is_province_record,province,period_label,production_original,year,period,quarter,value_status,production_mt
0,ALL SPECIES,ALL SPECIES,....Sulu,Sulu,True,Sulu,2002 Quarter 1,..,2002,Quarter 1,1.0000,unavailable,NaN
1,ALL SPECIES,ALL SPECIES,....Zamboanga del Norte,Zamboanga del Norte,True,Zamboanga del Norte,2002 Quarter 1,31001.00,2002,Quarter 1,1.0000,reported,"31,001.0000"
2,ALL SPECIES,ALL SPECIES,....Zamboanga del Sur,Zamboanga del Sur,True,Zamboanga del Sur,2002 Quarter 1,13925.00,2002,Quarter 1,1.0000,reported,"13,925.0000"
3,ALL SPECIES,ALL SPECIES,....Zamboanga Sibugay,Zamboanga Sibugay,True,Zamboanga Sibugay,2002 Quarter 1,11099.90,2002,Quarter 1,1.0000,reported,"11,099.9000"
4,ALL SPECIES,ALL SPECIES,....Camiguin,Camiguin,True,Camiguin,2002 Quarter 1,663.00,2002,Quarter 1,1.0000,reported,663.0000


## 4. Inspect source columns and coverage

In [ ]:
print("NASA columns:")
print(nasa.columns.tolist())

print("\\nPSA columns:")
print(psa.columns.tolist())

print("\\nNASA period types:")
display(nasa["period_type"].value_counts(dropna=False))

print("\\nPSA reporting statuses:")
display(psa["value_status"].value_counts(dropna=False))

print("\\nNASA provinces:", nasa["province"].nunique())
print("PSA provinces:", psa["province"].nunique())
print("PSA species categories:", psa["species"].nunique())

NASA columns:
['period_type', 'period', 'year', 'quarter', 'month', 'province_id', 'province', 'chlor_a_mean', 'chlor_a_valid_pixels', 'chlor_a_mask_pixels', 'sst_mean', 'sst_valid_pixels', 'sst_mask_pixels', 'chlor_a_months_observed', 'sst_months_observed']
\nPSA columns:
['Species', 'species', 'Geolocation', 'geolocation_label', 'is_province_record', 'province', 'period_label', 'production_original', 'year', 'period', 'quarter', 'value_status', 'production_mt']
\nNASA period types:


period_type
monthly      6888
quarterly    2304
Name: count, dtype: int64

\nPSA reporting statuses:


value_status
reported                     118599
unavailable                   90534
below_reporting_precision       242
Name: count, dtype: int64

\nNASA provinces: 24
PSA provinces: 24
PSA species categories: 67


## 5. Prepare quarterly NASA observations

The NASA source contains both monthly and quarterly records. We retain only `period_type == quarterly`, then keep the provincial quarterly means for chlorophyll-a and SST.

In [ ]:
nasa_q = nasa.loc[
    nasa["period_type"].astype(str).str.lower().eq("quarterly")
].copy()

nasa_q["Year"] = pd.to_numeric(nasa_q["year"], errors="coerce").astype("Int64")
nasa_q["Quarter"] = nasa_q["quarter"].astype(str).str.strip().str.upper()
nasa_q["Province"] = nasa_q["province"].astype(str).str.strip()

nasa_q = nasa_q[[
    "Year",
    "Quarter",
    "Province",
    "chlor_a_mean",
    "sst_mean"
]].rename(columns={
    "chlor_a_mean": "Chlorophyll-A Concentration (mg/m³)",
    "sst_mean": "Sea Surface Temperature (°C)"
})

print("Quarterly NASA records:", len(nasa_q))
print("Duplicate NASA province-quarter keys:",
      nasa_q.duplicated(["Year", "Quarter", "Province"]).sum())

display(nasa_q.head())

Quarterly NASA records: 2304
Duplicate NASA province-quarter keys: 0


,Year,Quarter,Province,Chlorophyll-A Concentration (mg/m³),Sea Surface Temperature (°C)
3,2002,Q3,Zamboanga del Norte,0.2463,29.4060
7,2002,Q4,Zamboanga del Norte,0.8728,29.2537
11,2003,Q1,Zamboanga del Norte,1.7041,26.4307
15,2003,Q2,Zamboanga del Norte,0.8020,29.1599
19,2003,Q3,Zamboanga del Norte,0.3305,29.8325


## 6. Prepare quarterly PSA fisheries production

Annual PSA records are excluded because the environmental variables are quarterly. We also exclude `ALL SPECIES`, which is an aggregate rather than an individual species/category.

In [ ]:
psa_q = psa.loc[psa["quarter"].notna()].copy()

psa_q["Year"] = pd.to_numeric(psa_q["year"], errors="coerce").astype("Int64")
psa_q["Quarter"] = (
    "Q" + pd.to_numeric(psa_q["quarter"], errors="coerce")
             .astype("Int64").astype(str)
)
psa_q["Province"] = psa_q["province"].astype(str).str.strip()
psa_q["Species"] = psa_q["species"].astype(str).str.strip()

# Exclude aggregate total records.
psa_q = psa_q.loc[
    psa_q["Species"].str.upper().ne("ALL SPECIES")
].copy()

print("Quarterly PSA records after excluding ALL SPECIES:", len(psa_q))
display(psa_q.head())

Quarterly PSA records after excluding ALL SPECIES: 165000


,Species,species,Geolocation,geolocation_label,is_province_record,province,period_label,production_original,year,period,quarter,value_status,production_mt,Year,Quarter,Province
25,Acetes (Alamang),Acetes (Alamang),....Sulu,Sulu,True,Sulu,2002 Quarter 1,..,2002,Quarter 1,1.0000,unavailable,NaN,2002,Q1,Sulu
26,Acetes (Alamang),Acetes (Alamang),....Zamboanga del Norte,Zamboanga del Norte,True,Zamboanga del Norte,2002 Quarter 1,19.00,2002,Quarter 1,1.0000,reported,19.0000,2002,Q1,Zamboanga del Norte
27,Acetes (Alamang),Acetes (Alamang),....Zamboanga del Sur,Zamboanga del Sur,True,Zamboanga del Sur,2002 Quarter 1,..,2002,Quarter 1,1.0000,unavailable,NaN,2002,Q1,Zamboanga del Sur
28,Acetes (Alamang),Acetes (Alamang),....Zamboanga Sibugay,Zamboanga Sibugay,True,Zamboanga Sibugay,2002 Quarter 1,60.00,2002,Quarter 1,1.0000,reported,60.0000,2002,Q1,Zamboanga Sibugay
29,Acetes (Alamang),Acetes (Alamang),....Camiguin,Camiguin,True,Camiguin,2002 Quarter 1,..,2002,Quarter 1,1.0000,unavailable,NaN,2002,Q1,Camiguin


## 7. Standardize species names

The PSA labels include local-language names in parentheses. These are removed so the final `Species` column uses the English label. A small explicit mapping handles the abbreviation `P. Vannamei`.

In [ ]:
def clean_species(value):
    raw = re.sub(r"\s+", " ", str(value)).strip()

    # Preserve distinct PSA categories before removing aliases.
    special = {
        "P. Vannamei": "Pacific White Shrimp",
        "Mullet (Kapak)": "Kapak Mullet",
        "Mullet (Ludong)": "Ludong Mullet",
        "Others (Aquaculture)": "Other Aquaculture Species",
        "Others (Marine Fisheries)": "Other Marine Fisheries Species",
    }

    if raw in special:
        return special[raw]

    # Remove parenthetical local-language names.
    value = re.sub(r"\s*\([^)]*\)", "", raw).strip()

    # Normalize whitespace.
    value = re.sub(r"\s+", " ", value)

    return value.strip(" .").title()

psa_q["Species"] = psa_q["Species"].map(clean_species)

print("Cleaned species categories:", psa_q["Species"].nunique())
display(pd.Series(sorted(psa_q["Species"].dropna().unique()),
                  name="Standardized Species").to_frame())

Cleaned species categories: 66


,Standardized Species
0,Acetes
1,Anchovies
2,Bali Sardinella
3,Big-Eyed Scad
4,Bigeye Tuna
...,...
61,Threadfin Bream
62,Tiger Prawn
63,Tilapia
64,White Shrimp


## 8. Resolve duplicate PSA records

Some PSA keys have both an `unavailable` row and a usable row. Resolve these by preferring `reported`, then `below_reporting_precision`, then `unavailable`.

A `below_reporting_precision` value of `0.00` remains zero. An `unavailable` value remains missing; it is not assumed to mean zero.

In [ ]:
priority = {
    "reported": 0,
    "below_reporting_precision": 1,
    "unavailable": 2
}

psa_q["_priority"] = (
    psa_q["value_status"].map(priority).fillna(3)
)

dedupe_keys = ["Year", "Quarter", "Province", "Species"]

# Check whether any duplicate keys contain conflicting reported values.
reported = psa_q.loc[psa_q["value_status"].eq("reported")]
reported_conflicts = (
    reported.groupby(dedupe_keys)["production_mt"]
            .nunique()
            .gt(1)
)

if reported_conflicts.any():
    raise ValueError(
        "Conflicting reported PSA values exist for the same "
        "Year/Quarter/Province/Species key. Inspect before deduplication."
    )

duplicate_rows_before = psa_q.duplicated(dedupe_keys, keep=False).sum()

psa_q = (
    psa_q.sort_values("_priority")
         .drop_duplicates(dedupe_keys, keep="first")
         .copy()
)

print("Rows involved in duplicate keys before resolution:",
      duplicate_rows_before)
print("PSA rows after duplicate resolution:", len(psa_q))
print("Remaining duplicate keys:",
      psa_q.duplicated(dedupe_keys).sum())

Rows involved in duplicate keys before resolution: 13200
PSA rows after duplicate resolution: 158400
Remaining duplicate keys: 0


## 9. Keep the required PSA fields

In [ ]:
psa_clean = psa_q[
    dedupe_keys + ["production_mt"]
].rename(columns={
    "production_mt": "Yield (Metric Tons)"
})

display(psa_clean.head())

,Year,Quarter,Province,Species,Yield (Metric Tons)
35,2002,Q1,Davao del Sur,Acetes,38.0000
205994,2026,Q3,Surigao del Sur,Snail,0.6800
205991,2026,Q3,Agusan del Norte,Snail,110.5000
205990,2026,Q3,Sultan Kudarat,Snail,0.2000
205987,2026,Q3,Davao Oriental,Snail,0.5200


## 10. Integrate NASA and PSA datasets

Join on `Year`, `Quarter`, and `Province`. The join is many-to-one: one NASA provincial-quarter observation can correspond to multiple PSA species records.

In [ ]:
clean = psa_clean.merge(
    nasa_q,
    on=["Year", "Quarter", "Province"],
    how="inner",
    validate="many_to_one"
)

final_columns = [
    "Year",
    "Quarter",
    "Province",
    "Species",
    "Chlorophyll-A Concentration (mg/m³)",
    "Sea Surface Temperature (°C)",
    "Yield (Metric Tons)"
]

clean = (
    clean[final_columns]
    .sort_values(["Year", "Quarter", "Province", "Species"])
    .reset_index(drop=True)
)

print("Integrated dataset shape:", clean.shape)
display(clean.head())

Integrated dataset shape: (152064, 7)


,Year,Quarter,Province,Species,Chlorophyll-A Concentration (mg/m³),Sea Surface Temperature (°C),Yield (Metric Tons)
0,2002,Q3,Agusan del Norte,Acetes,1.0101,30.1495,3.0000
1,2002,Q3,Agusan del Norte,Anchovies,1.0101,30.1495,210.0000
2,2002,Q3,Agusan del Norte,Bali Sardinella,1.0101,30.1495,83.0000
3,2002,Q3,Agusan del Norte,Big-Eyed Scad,1.0101,30.1495,30.0000
4,2002,Q3,Agusan del Norte,Bigeye Tuna,1.0101,30.1495,NaN


## 11. Validate the final dataset

Missing yield values are expected for PSA records marked unavailable. Do not impute them during this source-integration step. Chlorophyll-a or SST missingness should also be reported rather than silently filled.

In [ ]:
print("Final columns:")
print(clean.columns.tolist())

print("\\nData types:")
display(clean.dtypes.to_frame("dtype"))

print("\\nMissing values:")
display(clean.isna().sum().to_frame("missing_count"))

print("\\nQuarter values:")
display(clean["Quarter"].value_counts(dropna=False).sort_index())

print("\\nYear range:", clean["Year"].min(), "to", clean["Year"].max())
print("Province count:", clean["Province"].nunique())
print("Species count:", clean["Species"].nunique())

# Structural checks.
assert clean.columns.tolist() == final_columns
assert clean["Year"].notna().all()
assert clean["Quarter"].isin(["Q1", "Q2", "Q3", "Q4"]).all()
assert clean["Province"].notna().all()
assert clean["Species"].notna().all()
assert not clean.duplicated(
    ["Year", "Quarter", "Province", "Species"]
).any()

# Check that all environmental values are numeric.
for col in [
    "Chlorophyll-A Concentration (mg/m³)",
    "Sea Surface Temperature (°C)",
    "Yield (Metric Tons)"
]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

print("\\nAll structural validation checks passed.")

Final columns:
['Year', 'Quarter', 'Province', 'Species', 'Chlorophyll-A Concentration (mg/m³)', 'Sea Surface Temperature (°C)', 'Yield (Metric Tons)']
\nData types:


,dtype
Year,Int64
Quarter,str
Province,str
Species,str
Chlorophyll-A Concentration (mg/m³),float64
Sea Surface Temperature (°C),float64
Yield (Metric Tons),float64


\nMissing values:


,missing_count
Year,0
Quarter,0
Province,0
Species,0
Chlorophyll-A Concentration (mg/m³),0
Sea Surface Temperature (°C),0
Yield (Metric Tons),62618


\nQuarter values:


Quarter
Q1    38016
Q2    38016
Q3    38016
Q4    38016
Name: count, dtype: int64

\nYear range: 2002 to 2026
Province count: 24
Species count: 66
\nAll structural validation checks passed.


**Check 1: Yield availability by year**

In [ ]:
yield_by_year = (
    clean.groupby("Year")["Yield (Metric Tons)"]
    .agg(
        total_records="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

yield_by_year["missing_percentage"] = (
    yield_by_year["missing"] /
    yield_by_year["total_records"] * 100
)

display(yield_by_year)

,total_records,available,missing,missing_percentage
Year,,,,
2002,3168,1441,1727,54.5139
2003,6336,3043,3293,51.9729
2004,6336,3299,3037,47.9324
2005,6336,3577,2759,43.5448
2006,6336,3677,2659,41.9665
2007,6336,3726,2610,41.1932
2008,6336,3778,2558,40.3725
2009,6336,3769,2567,40.5145
2010,6336,3763,2573,40.6092


**Check 2: Yield availability by species**

In [ ]:
yield_by_species = (
    clean.groupby("Species")["Yield (Metric Tons)"]
    .agg(
        total_records="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

yield_by_species["missing_percentage"] = (
    yield_by_species["missing"] /
    yield_by_species["total_records"] * 100
)

display(
    yield_by_species.sort_values(
        "missing_percentage",
        ascending=False
    )
)

,total_records,available,missing,missing_percentage
Species,,,,
Sardines,2304,51,2253,97.7865
Mussel,2304,104,2200,95.4861
Ludong Mullet,2304,124,2180,94.6181
Starry Goby,2304,220,2084,90.4514
Spiny Lobster,2304,278,2026,87.9340
...,...,...,...,...
Grouper,2304,2149,155,6.7274
Indian Mackerel,2304,2152,152,6.5972
Roundscad,2304,2175,129,5.5990


**Check 3: Environmental consistency**

Verify that each province-quarter has exactly one environmental measurement, regardless of species.

In [ ]:
environment_check = (
    clean.groupby(["Year", "Quarter", "Province"])
    .agg(
        chlorophyll_values=(
            "Chlorophyll-A Concentration (mg/m³)", "nunique"
        ),
        sst_values=(
            "Sea Surface Temperature (°C)", "nunique"
        )
    )
)

print("Multiple chlorophyll measurements:",
      (environment_check["chlorophyll_values"] > 1).sum())

print("Multiple SST measurements:",
      (environment_check["sst_values"] > 1).sum())

Multiple chlorophyll measurements: 0
Multiple SST measurements: 0


## 12. Export the final clean CSV

In [ ]:
clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_FILE.resolve()}")
print(f"Rows: {len(clean):,}")
print(f"Columns: {len(clean.columns)}")

# Read the exported file back to verify the saved output.
check = pd.read_csv(OUTPUT_FILE)
assert check.columns.tolist() == final_columns
assert check.shape == clean.shape

print("CSV export verified.")
display(check.head())

Saved: D:\blueshift\data\processed\blueshift_dataset.csv
Rows: 152,064
Columns: 7
CSV export verified.


,Year,Quarter,Province,Species,Chlorophyll-A Concentration (mg/m³),Sea Surface Temperature (°C),Yield (Metric Tons)
0,2002,Q3,Agusan del Norte,Acetes,1.0101,30.1495,3.0000
1,2002,Q3,Agusan del Norte,Anchovies,1.0101,30.1495,210.0000
2,2002,Q3,Agusan del Norte,Bali Sardinella,1.0101,30.1495,83.0000
3,2002,Q3,Agusan del Norte,Big-Eyed Scad,1.0101,30.1495,30.0000
4,2002,Q3,Agusan del Norte,Bigeye Tuna,1.0101,30.1495,NaN


## Final notes for EDA and model training

- The dataset is at **province-quarter-species** level.
- NASA environmental values repeat across species within the same province-quarter. They are not species-specific measurements.
- `Yield (Metric Tons)` is the target candidate. Rows with missing yield represent unavailable PSA production, not zero production.
- Excluding `ALL SPECIES` prevents aggregate totals from being mixed with species-level records.
- For model training, split data chronologically where appropriate and ensure future quarters do not leak into training features.
- Consider whether the modeling objective is species-level yield prediction or total provincial fisheries production; these require different aggregation strategies.